# ANÁLISE DO DESENVOLVIMENTO HUMANO E ECONÔMICO GLOBAL

**Grupo:** André Marcus, Gabriel Moreira, Leandro Mendes Thiago Leite

## 1. Título e Introdução

### Tema do projeto



### Objetivos da análise



### Contexto da base de dados


### Etapas realizadas no notebook

O desenvolvimento do projeto será organizado nas seguintes etapas:

1. **Coleta e integração dos dados:** obtenção da base Gapminder e dos indicadores complementares do Banco Mundial.
2. **Tratamento dos dados:** identificação de valores ausentes e duplicados, limpeza e organização das informações.
3. **Análise exploratória:** investigação dos indicadores por meio de tabelas, estatísticas e gráficos.
4. **Criação de funções reutilizáveis:** desenvolvimento de funções para facilitar a construção e personalização das visualizações.
5. **Interpretação dos resultados:** análise dos padrões identificados e das relações entre os indicadores.
6. **Síntese e conclusão:** apresentação dos principais insights e considerações finais sobre o desenvolvimento humano e econômico global.

### Importação das bibliotecas

In [9]:
%pip install scipy

In [19]:
# Importar as bibliotecas que serão utilizadas pelo grupo
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import csv
from charset_normalizer import from_bytes #Para detectar encoding e separador dos arquivos automaticamente
import urllib.request as ur   #Para aceitar base de dados que venham de URL



---

## 2. Coleta, Limpeza e Análise Exploratória (AED)

**Objetivo:** preparar os dados, tratando inconsistências e identificando padrões iniciais. Esta seção está organizada por fonte de dados:
- **2.1 Gapminder:** coleta, exploração inicial, limpeza e transformação.
- **2.2 Bases externas (Banco Mundial):** preparação dos indicadores externos e junção com o Gapminder limpo. Cada tema tem a sua subseção (2.2.1 Economia).

### 2.1 Gapminder

#### 2.1.1 Coleta de Dados

O primeiro passo é carregar a base Gapminder e registrar sua origem. Confiram se o arquivo foi disponibilizado junto com o trabalho e anotem a fonte utilizada.

In [12]:
# Informar o caminho correto do arquivo disponibilizado para a atividade
df = pd.read_csv('../gapminder_full.csv')

df.head()


FileNotFoundError: [Errno 2] No such file or directory: '../gapminder_full.csv'

In [ ]:
df.shape

#### 2.1.2 Exploração Inicial dos Dados

Antes de tratar os dados, verificar a estrutura do DataFrame e entender os itens em cada colunas.

In [ ]:
# Consultar estatísticas descritivas

df.info()

df.describe().round(2)

In [ ]:
# Verificar os países, continentes e anos presentes na base
print('Quantidade de países:', df['country'].nunique())
print('Continentes:', df['continent'].unique())
print('Anos:', df['year'].unique())

In [ ]:
# Quantidade de países em cada continente
df.groupby('continent')['country'].nunique()

In [ ]:
# (se cada país aparecesse só uma vez, seriam 142 linhas por ano)
df['year'].value_counts().sort_index()

**Anotações**
- **Colunas disponíveis:** `country` (país), `year` (ano), `pop` (população), `continent` (continente), `lifeExp` (expectativa de vida) e `gdpPercap` (PIB per capita). São 6 colunas e 1.736 linhas.
- **Período coberto:** de 1952 a 2007, de 5 em 5 anos (12 anos no total).
- **Países e continentes:** 142 países em 5 continentes. A Oceania tem só 2 países, então a média dela deve ser vista com cuidado.
- **O que verificar antes da limpeza:**
  - Cada ano tem entre 143 e 148 linhas, mas o esperado era 142 (um por país). Isso indica que possivelmente existem linhas **duplicadas**.
  - A coluna `pop` está como número decimal (`float64`), mas população é uma contagem de pessoas.

#### 2.1.3 Limpeza de Dados

A atividade solicita excluir os dados ausentes e eliminar registros duplicados. Antes e depois da limpeza, registrem a quantidade de linhas e as decisões tomadas.

In [ ]:
# Verificar a quantidade de valores ausentes por coluna | Limpeza abaixo
print('Linhas antes da limpeza:', len(df))
df.isna().sum()

df = df.dropna()
print('Linhas após a limpeza dos valores nulos:', len(df))

In [ ]:
# Verificar a quantidade de registros duplicados
print(f'Linhas duplicadas: {df.duplicated().sum()}')
print('Abaixo há exmeplos de linhas duplicadas. É possivel verificar essas linhas no arquivo completo do "gapminder_full.csv"')

df[df.duplicated()].head(10)

In [ ]:
# Eliminar registros duplicados e corrigindo os indices conforme ensinado
print('Linhas antes da limpeza:', len(df))
df = df.drop_duplicates()

print('Linhas após a limpeza dos duplicados:', len(df))
df = df.reset_index(drop=True)

In [ ]:
# Conferir a base após a limpeza

print('Linhas depois da limpeza:', len(df))
print('Duplicados restantes:', df.duplicated().sum())
print('Ausentes restantes:', df.isna().sum().sum())
df['year'].value_counts().sort_index()

**Decisões de tratamento**
- **Valores ausentes:** nenhum foi encontrado. Mantendo a mesma quantidade de linhas antes e após a limpeza.
- **Duplicados:** foram encontradas e removidas (32 linhas duplicadas).
- **Registros que restaram:** de 1.736 para 1.704 linhas, que é 142 países × 12 anos. Da forma que era esperado na etapa 2

#### 2.1.4 Transformação e Preparação dos Dados

Avaliando se é necessário ajustar tipos de dados, nomes de colunas ou outras tarefas para facilitar as análises.

In [ ]:
# Vê quais dados poderm ser alterados e converte a população para inteiros
df.info()

df['pop'] = df['pop'].round().astype('int64')
df.head()

In [ ]:
# Cria uma base de dados limpa que vai ser utilizada para fazer as análises
df.to_csv('../data/gapminder_limpo.csv', index=False)

### 2.2 Bases Externas (Banco Mundial)

Para responder perguntas sobre inflação, comércio, desemprego e diferença de expectativa de vida entre homens e mulheres, será utilizado bases do "Banco Mundial (World Development Indicators)", baixadas em https://data.worldbank.org.

Aqui é feito a preparação dos dados para comn=binar com o gapMinder que limpamos:
1. Ler os arquivos baixados.
2. Arrumar o formato da tabela (uma linha por país e ano).
3. Conferir se os nomes dos países batem com os do Gapminder.
4. Juntar com o `df` (Gapminder limpo) usando `country` e `year`.
5. Ver quantos dados ausentes sobraram.

#### 2.2.1 Economia (André)

Bases externas que será usada para analise de perguntas relacionadas a economia:

| Nome da coluna | Indicador do Banco Mundial | Unidade |
|---|---|---|
| `inflacao` | Inflação, preços ao consumidor | % ao ano |
| `comercio` | Comércio (exportações + importações) | % do PIB |
| `cresc_pib` | Crescimento do PIB per capita | % ao ano |
| `desemprego` | Desemprego total (estimativa modelada da OIT) | % da força de trabalho |
| `vida_f` | Expectativa de vida ao nascer, mulheres | anos |
| `vida_m` | Expectativa de vida ao nascer, homens | anos |

In [ ]:
# Organizando os caminhos para os arquivos, já que serão usados em algumas analises.

pasta = '../data/Economia/'

arquivos_economia = {
    'inflacao':   'API_FP.CPI.TOTL.ZG_DS2_en_csv_v2_1323.csv',
    'comercio':   'API_NE.TRD.GNFS.ZS_DS2_en_csv_v2_1202.csv',
    'cresc_pib':  'API_NY.GDP.PCAP.KD.ZG_DS2_en_csv_v2_6324.csv',
    'desemprego': 'API_SL.UEM.TOTL.ZS_DS2_en_csv_v2_1675.csv',
    'vida_f':     'API_SP.DYN.LE00.FE.IN_DS2_en_csv_v2_7315.csv',
    'vida_m':     'API_SP.DYN.LE00.MA.IN_DS2_en_csv_v2_3426.csv',
}

***Padronizando e lendo os arquivos***

O Gapminder e as fontes de dados vindo do WorldBank possuem uma estrutura diferente, então vai ser criado uma função simples, para não repetir o mesmo código seis vezes.
O 'melt' é usado para padronizar as colunas de anos em linhas, para facilitar a leitura, já os arquivos são diferentes

In [ ]:
def ler_indicador(arquivo, nome_coluna):

    tabela = pd.read_csv(pasta + arquivo, skiprows=4)

    # Colunas que não iremos usar
    tabela = tabela.drop(columns=['Country Code', 'Indicator Name', 'Indicator Code', 'Unnamed: 70'],
                         errors='ignore')


    tabela = tabela.melt(id_vars='Country Name', var_name='year', value_name=nome_coluna)


    tabela = tabela.rename(columns={'Country Name': 'country'})
    tabela['year'] = tabela['year'].astype(int)

    return tabela

***Junta os indicadores***
Juntando os indicadores para estarem em uma tabela

In [ ]:
# Começa vazia e vai juntando um indicador por vez
wb_economia = None

for nome, arquivo in arquivos_economia.items():
    tabela = ler_indicador(arquivo, nome)

    if wb_economia is None:
        wb_economia = tabela
    else:
        wb_economia = wb_economia.merge(tabela, on=['country', 'year'], how='outer')

print('Linhas e colunas:', wb_economia.shape)
wb_economia.head()

Como tem uma descrepancia no nome dos paises entre essas duas fontes, vamos verificar e modificar as que conseguimos

In [ ]:
# Países do Gapminder que não aparecem no WorldBank
paises_gapminder = set(df['country'].unique())
paises_wb = set(wb_economia['country'].unique())

nao_encontrados = sorted(paises_gapminder - paises_wb)
print('Quantidade:', len(nao_encontrados))
print(nao_encontrados)

In [ ]:
# Corrigindo o nome de paises do World Bank para o nome no Gapminder
correcao_nomes = {
    'Congo, Dem. Rep.': 'Congo Dem. Rep.',
    'Congo, Rep.': 'Congo Rep.',
    'Czechia': 'Czech Republic',
    'Egypt, Arab Rep.': 'Egypt',
    'Gambia, The': 'Gambia',
    'Hong Kong SAR, China': 'Hong Kong China',
    'Iran, Islamic Rep.': 'Iran',
    "Korea, Dem. People's Rep.": 'Korea Dem. Rep.',
    'Korea, Rep.': 'Korea Rep.',
    'Puerto Rico (US)': 'Puerto Rico',
    'Somalia, Fed. Rep.': 'Somalia',
    'Eswatini': 'Swaziland',
    'Syrian Arab Republic': 'Syria',
    'Turkiye': 'Turkey',
    'Venezuela, RB': 'Venezuela',
    'Viet Nam': 'Vietnam',
    'Yemen, Rep.': 'Yemen Rep.',
}

wb_economia['country'] = wb_economia['country'].replace(correcao_nomes)

# Conferindo de novo quais países continuam sem correspondência
paises_wb = set(wb_economia['country'].unique())
print('Ainda não encontrados:', sorted(paises_gapminder - paises_wb))

In [ ]:
# Junta o Gapminder limpo com os indicadores do WB depois remove ausentes
df_economia = df.merge(wb_economia, on=['country', 'year'], how='left')

print('Linhas antes:', len(df))
print('Linhas depois:', len(df_economia))
print('Duplicados em país/ano:', df_economia.duplicated(['country', 'year']).sum())

df_economia.head()



In [ ]:
df_economia.isna().sum()
colunas_wb = list(arquivos_economia.keys())

df_economia.groupby('year')[colunas_wb].count()

### 2.3 Saúde (Leandro)

Nesta parte eu trago quatro base de dados novas para cruzar com a expectativa de vida e o PIB
per capita do Gapminder. Sendo uma base de dados nova para cada pergunta a ser respondida, sendo elas:   

1. Países que gastam mais em saúde per capita têm maior expectativa de vida, mesmo quando o PIB per capita é parecido?
2. A mortalidade infantil explica melhor as diferenças de expectativa de vida entre países
        ricos e pobres do que o PIB per capita?
3. Países com mais veículos por habitante têm melhor longevidade
4. A taxa de obesidade cresce junto com o PIB per capita?
5. Países com mais médicos por mil habitantes têm maior expectativa de vida, mesmo descontando o efeito do PIB per capita?


#### 2.3.1 Coleta de Dados

**#Importa bases de dados**


In [13]:
# Substitua a URL abaixo pela URL 'raw' do seu arquivo no GitHub
url_github =['https://raw.githubusercontent.com/AndremRb/Analise-do-Desenvolvimento-Humano-e-Global/refs/heads/main/gapminder_full.csv',
              'https://raw.githubusercontent.com/AndremRb/Analise-do-Desenvolvimento-Humano-e-Global/refs/heads/main/Leandro%20(An%C3%A1lise%20explorat%C3%B3ria%20Sa%C3%BAde)/bases_externas/gasto_saude_per_capita.csv',
              'https://raw.githubusercontent.com/AndremRb/Analise-do-Desenvolvimento-Humano-e-Global/refs/heads/main/Leandro%20(An%C3%A1lise%20explorat%C3%B3ria%20Sa%C3%BAde)/bases_externas/medicos_por_1000_habitantes.csv',
              'https://raw.githubusercontent.com/AndremRb/Analise-do-Desenvolvimento-Humano-e-Global/refs/heads/main/Leandro%20(An%C3%A1lise%20explorat%C3%B3ria%20Sa%C3%BAde)/bases_externas/mortalidade_infantil.csv',
              'https://raw.githubusercontent.com/AndremRb/Analise-do-Desenvolvimento-Humano-e-Global/refs/heads/main/Leandro%20(An%C3%A1lise%20explorat%C3%B3ria%20Sa%C3%BAde)/bases_externas/taxa_obesidade.csv',
              'https://raw.githubusercontent.com/AndremRb/Analise-do-Desenvolvimento-Humano-e-Global/refs/heads/main/Leandro%20(An%C3%A1lise%20explorat%C3%B3ria%20Sa%C3%BAde)/bases_externas/veiculos_por_1000_habitantes.csv']


**#Detecta o tipo de Encoding e Separador de CSVs**


In [14]:
def configuraçoes_csv(caminho_arquivo):
    """
    Descobre o encoding e o separador de qualquer CSV de forma rápida e certeira.
    Aceita tanto arquivos locais quanto links da internet (ex: GitHub).
    Retorna: (encoding, separador).
    """

    # 1. Verifica se é um link da internet ou arquivo local
    if caminho_arquivo.startswith('http://') or caminho_arquivo.startswith('https://'):   # Verifica se começa com um dos termos
        # Se for link, faz uma requisição HTTP e lê os primeiros 20.000 bytes
        req = ur.Request(
            caminho_arquivo,
            headers={'User-Agent': 'Mozilla/5.0'} # Simula um cabeçalho real na requisição como se fosse um usuário acessando pelo Mozila
        )
        with ur.urlopen(req) as f:   # Abre a conexão com o servidor através da requisição anterior criada
            amostra_binaria = f.read(20000)   #Lê e armazena os primeiros 20000 bytes em amostra_binária
                                              #NOTA: não usamos open(caminho_arquivo, "rb") pq na internet tudo já está em byte
    else:
        # Se for arquivo local, abre normalmente
        with open(caminho_arquivo, 'rb') as f:    #Aqui usamos o 'rb' pois arquivos locais não costuman estar em binário pois nós não conseguiriamos ler.
            amostra_binaria = f.read(20000)

    # 2. Descobre o encoding com quase 100% de precisão
    resultado = from_bytes(amostra_binaria).best()  # Analisa a amostra binária
    encoding = resultado.encoding   # Armazena o nome do encoding detectado
    texto_amostra = str(resultado)    # Converte em texto legível novamente

    # 3. Descobre o separador (.csv, .txt, etc)
    dialeto = csv.Sniffer().sniff(texto_amostra)    #"Fareja" em nossa amostra qual o separador mais aparece e
    separador = dialeto.delimiter

    return caminho_arquivo,encoding, separador

**#Cria um único DataFrame**

In [15]:
def cria_dataframe(caminho_arquivo,encoding,separador):
    df=pd.read_csv(caminho_arquivo, encoding=encoding, sep=separador)
    return df


**#Cria múltiplos DataFrames recebendo lista de URLs ou CSVs**

In [16]:
def carrega_multiplos_dataframes(lista_caminhos):
    """
    Recebe uma lista de URLs ou caminhos locais e retorna uma lista de DataFrames carregados
    automaticamente usando as configurações detectadas para cada um deles.

    **Exemplo de entrada:**
    lista caminhos= [url1, url2, url3, ...] ou [caminho1, caminho2, caminho3, ...]

    **Exemplo de saída:**
    dataframes= [df1, df2, df3, ...]
    """
    dataframes = []   #cria uma lista vazia para nosso append ir add oq precisa
    for caminho in lista_caminhos:
        # 1. Detecta as configurações do arquivo atual
        config = configuraçoes_csv(caminho)
        # 2. Cria o DataFrame correspondente
        df = cria_dataframe(*config)#garante que cada termo da tupla seja descompactado e utilizado na função
        # 3. Adiciona o DataFrame à lista
        dataframes.append(df)
    return dataframes

# Exemplo de uso prático:
# lista_de_fontes = [url_github]  # Você pode colocar quantos links ou caminhos quiser aqui
# meus_dfs = carregar_multiplos_dataframes(lista_de_fontes)
# meus_dfs[0].head()  # Acessa o primeiro DataFrame gerado

**#Apresenta um DICIONÁRIO de nomes para cada Dataframe na lista**

In [29]:
nomes = ['gapminder', 'saude', 'medicos', 'mortalidade', 'obesidade', 'veiculos'] #Cria uma lista de nomes para nossas bases de dados
dataframes_copia = carrega_multiplos_dataframes(url_github).copy() #Copia nossa lista de tabelas para comecarmos o tratamento de dados
df_brutos = dict(zip(nomes, dataframes_copia)) #Cria Dicionários e nomeia as tabelas com os nomes da lista acima


**#Função Superior para aplicar outras funções em cada um dos Dataframes de uma lista de tabelas**


In [21]:
def aplicar_funcoes(dicionario, funcao, decisor):
  """
  Aplica uma função a todas as tabelas de um dicionário {nome: tabela}.

  Parâmetros:
    dicionario : dicionário {nome: DataFrame} a processar.
    funcao     : função a ser aplicada em cada tabela.
    decisor    : 1 → transformar (sobrescreve as tabelas com o resultado da função);
                 0 → visualizar/investigar (apenas executa, etiquetando pelo nome; não altera);
                 2 → coletar (não altera o original; devolve um dicionário novo {nome: resultado}).

  Retorno:
    decisor=1 → dicionário com as tabelas transformadas;
    decisor=0 → o dicionário original (inalterado);
    decisor=2 → dicionário novo {nome: resultado da função}.
  """
  if decisor not in (0, 1, 2):
    raise ValueError("decisor deve ser 0 (visualizar), 1 (transformar) ou 2 (coletar)")

  if decisor == 1:                                   # TRANSFORMAR
    dic_alterado = dicionario.copy()
    for nome in dic_alterado:
        dic_alterado[nome] = funcao(dic_alterado[nome])
    return dic_alterado

  elif decisor == 2:                                 # COLETAR
    resultados = {}
    for nome, tabela in dicionario.items():
        resultados[nome] = funcao(tabela)
    return resultados

  else:                                              # VISUALIZAR (decisor == 0)
    for nome, tabela in dicionario.items():
        print(f'\n===== {nome} =====')
        funcao(tabela)
    return dicionario

#### 2.3.2 Limpeza de Dados

Confiro valores duplicados e ausentes. Removo os duplicados; os ausentes eu deixo como estão aqui,
porque cada indicador tem anos diferentes — na hora de cada análise eu uso só as colunas necessárias
(as funções de gráfico já descartam os valores vazios).

#### 2.3.2.0 Limpeza de Dados (Base Gapminder)

**#Exploracao inicial**

Aqui entendemos a estrutura da tabela:

1. country: País
2. year: o Ano do dado
3. pop: população total do país naquele ano
4. continent: Continente em que o país fica
5. lifeExp: expectativa de vida ao nascer (em anos) naquele país e ano
6. gdpPercap: PIB per capita (em dólares internacionais), ou seja, a riqueza média por pessoa

Aqui conhecemos um pouco da estrutura da tabela

In [49]:
df_brutos['gapminder'].tail(10).round(1)

,country,year,pop,continent,lifeExp,gdpPercap
1694,Zimbabwe,1962,4277736.0,Africa,52.4,527.3
1695,Zimbabwe,1967,4995432.0,Africa,54.0,569.8
1696,Zimbabwe,1972,5861135.0,Africa,55.6,799.4
1697,Zimbabwe,1977,6642107.0,Africa,57.7,685.6
1698,Zimbabwe,1982,7636524.0,Africa,60.4,788.9
1699,Zimbabwe,1987,9216418.0,Africa,62.4,706.2
1700,Zimbabwe,1992,10704340.0,Africa,60.4,693.4
1701,Zimbabwe,1997,11404948.0,Africa,46.8,792.4
1702,Zimbabwe,2002,11926563.0,Africa,40.0,672.0
1703,Zimbabwe,2007,12311143.0,Africa,43.5,469.7


Aqui entendemos quantidade de nulos
e o tipo de dado de cada coluna

In [22]:
df_brutos['gapminder'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1736 entries, 0 to 1735
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1736 non-null   object 
 1   year       1736 non-null   int64  
 2   pop        1736 non-null   float64
 3   continent  1736 non-null   object 
 4   lifeExp    1736 non-null   float64
 5   gdpPercap  1736 non-null   float64
dtypes: float64(3), int64(1), object(2)
memory usage: 81.5+ KB


Aqui Vizualizamos estatisticamente a distribuicao dos valores dentro dessa tabela

In [23]:
#Analize estatistica da tabela como um todo
df_brutos['gapminder'].describe().round(1)

,year,pop,lifeExp,gdpPercap
count,1736.0,1.736000e+03,1736.0,1736.0
mean,1979.7,3.156800e+07,59.5,7240.2
std,17.3,1.172888e+08,13.0,9839.9
min,1952.0,6.001100e+04,23.6,241.2
25%,1967.0,2.818694e+06,48.2,1202.1
50%,1982.0,7.150606e+06,60.8,3534.8
75%,1997.0,1.977102e+07,70.9,9380.3
max,2007.0,1.318683e+09,82.6,113523.1


Aqui Verificamos a quantidade de nulos e duplicados da base de dados para entendermos se faz necessaria uma limpeza ou não.

In [24]:
qtd_nulos = df_brutos['gapminder'].isna().sum()
print('Quantidade de nulos por  antes da limpeza:')
print(qtd_nulos)

Quantidade de nulos por  antes da limpeza:
country      0
year         0
pop          0
continent    0
lifeExp      0
gdpPercap    0
dtype: int64


In [25]:
qtd_duplicados = df_brutos['gapminder'].duplicated().sum()
print('Quantidade de duplicados antes da limpeza:', qtd_duplicados)

Quantidade de duplicados antes da limpeza: 32


In [39]:
#Aqui vizualizamos os duplicados
df_brutos_duplicados = df_brutos['gapminder'].duplicated(keep=False)

print('Linhas duplicadas:')
display(df_brutos['gapminder'][df_brutos_duplicados])

Linhas duplicadas:


,country,year,pop,continent,lifeExp,gdpPercap
9,Afghanistan,1997,22227415.0,Asia,41.763,635.341351
72,Afghanistan,1997,22227415.0,Asia,41.763,635.341351
120,Belgium,2007,10392226.0,Europe,79.441,33692.605080
140,Bolivia,1987,6156369.0,Americas,57.251,2753.691490
141,Bolivia,1992,6893451.0,Americas,59.957,2961.699694
...,...,...,...,...,...,...
644,Guinea,1952,2664249.0,Africa,33.609,510.196492
645,Guinea,1957,2876726.0,Africa,34.558,576.267025
646,Guinea,1962,3140003.0,Africa,35.753,686.373674
647,Guinea,1967,3451418.0,Africa,37.197,708.759541


Nessa etapa nós eliminamos os duplicados e NULL

In [41]:
df_brutos['gapminder'] = df_brutos['gapminder'].dropna()
df_brutos['gapminder'] = df_brutos['gapminder'].drop_duplicates()

Aqui verificamos a quantidade de nulos e duplicados, se de fato foram eliminados:

In [43]:
qtd_nulos = df_brutos['gapminder'].isna().sum()
print('Quantidade de nulos por  após a limpeza:')
print(qtd_nulos)

Quantidade de nulos por  após a limpeza:
country      0
year         0
pop          0
continent    0
lifeExp      0
gdpPercap    0
dtype: int64


In [44]:
qtd_duplicados = df_brutos['gapminder'].duplicated().sum()
print('Quantidade de duplicados após a limpeza:', qtd_duplicados)

Quantidade de duplicados após a limpeza: 0


Agora nós corrigimos o INDEX da tabela após eliminarmos algumas linhas .

In [47]:
df_brutos['gapminder'].reset_index(drop=True, inplace=True)

Agora verificamos o total de nomes unicos em cada coluna

In [46]:
df_brutos['gapminder'].nunique()

,0
country,142
year,12
pop,1704
continent,5
lifeExp,1626
gdpPercap,1704


#### 2.3.2.1 Limpeza de Dados(Base saude Per Capita)

**#Exploracao inicial**

Aqui entendemos a estrutura da tabela:

1. Entity: País
2. Code: Cóidigo do país
3. Year: o Ano do dado
4. Current health expenditure (Current international $): Em média, esse país gasta com a saúde de cada cidadão o equivalente ao que X dólares comprariam de saúde dentro dos Estados Unidos (uma forma de equiparar as moedas)

Aqui entendemos quantidade de nulos
e o tipo de dado de cada coluna

In [ ]:
df_brutos['saude'].info()


Aqui Vizualizamos estatisticamente a distribuicao dos valores dentro dessa tabela


In [ ]:
#Analize estatistica da tabela como um todo
df_brutos['saude'].describe().round(1)

Aqui Verificamos a quantidade de nulos e duplicados da base de dados para entendermos se faz necessaria uma limpeza ou não.

In [ ]:
qtd_nulos = df_brutos['saude'].isna().sum()
print('Quantidade de nulos por coluna:')
print(qtd_nulos)

In [ ]:
qtd_duplicados = df_brutos['saude'].duplicated().sum()
print('Quantidade de duplicados:', qtd_duplicados)

#### 2.3.2.2 Limpeza de Dados (Base Mortalidade Infantil)

**#Exploracao inicial**

Aqui entendemos a estrutura da tabela:

1. Entity: País
2. Code: Código do país
3. Year: o Ano do dado
4. Under-five mortality rate: a quantidade de crianças que morrem antes de completar 5 anos a cada 1.000 nascidas vivas naquele país e ano

Aqui entendemos quantidade de nulos
e o tipo de dado de cada coluna

In [ ]:
df_brutos['mortalidade'].info()

Aqui Vizualizamos estatisticamente a distribuicao dos valores dentro dessa tabela

In [ ]:
#Analize estatistica da tabela como um todo
df_brutos['mortalidade'].describe().round(1)

Aqui Verificamos a quantidade de nulos e duplicados da base de dados para entendermos se faz necessaria uma limpeza ou não.

In [ ]:
qtd_nulos = df_brutos['mortalidade'].isna().sum()
print('Quantidade de nulos por coluna:')
print(qtd_nulos)

In [ ]:
qtd_duplicados = df_brutos['mortalidade'].duplicated().sum()
print('Quantidade de duplicados:', qtd_duplicados)

#### 2.3.2.3 Limpeza de Dados (Base Obesidade)

**#Exploracao inicial**

Aqui entendemos a estrutura da tabela:

1. Entity: País
2. Code: Código do país
3. Year: o Ano do dado
4. Obesity among adults (%): a porcentagem de adultos (18 anos ou mais) com obesidade (IMC maior ou igual a 30) naquele país e ano
5. World region according to OWID: a região/continente do país, segundo o Our World in Data

Aqui entendemos quantidade de nulos
e o tipo de dado de cada coluna

In [ ]:
df_brutos['obesidade'].info()

Aqui Vizualizamos estatisticamente a distribuicao dos valores dentro dessa tabela

In [ ]:
#Analize estatistica da tabela como um todo
df_brutos['obesidade'].describe().round(1)

Aqui Verificamos a quantidade de nulos e duplicados da base de dados para entendermos se faz necessaria uma limpeza ou não.

In [ ]:
qtd_nulos = df_brutos['obesidade'].isna().sum()
print('Quantidade de nulos por coluna:')
print(qtd_nulos)

In [ ]:
qtd_duplicados = df_brutos['obesidade'].duplicated().sum()
print('Quantidade de duplicados:', qtd_duplicados)

#### 2.3.2.4 Limpeza de Dados (Base Médicos)

**#Exploracao inicial**

Aqui entendemos a estrutura da tabela:

1. Entity: País
2. Code: Código do país
3. Year: o Ano do dado
4. Physicians (per 1,000 people): a quantidade de médicos para cada 1.000 habitantes naquele país e ano

Aqui entendemos quantidade de nulos
e o tipo de dado de cada coluna

In [ ]:
df_brutos['medicos'].info()

Aqui Vizualizamos estatisticamente a distribuicao dos valores dentro dessa tabela

In [ ]:
#Analize estatistica da tabela como um todo
df_brutos['medicos'].describe().round(1)

Aqui Verificamos a quantidade de nulos e duplicados da base de dados para entendermos se faz necessaria uma limpeza ou não.

In [ ]:
qtd_nulos = df_brutos['medicos'].isna().sum()
print('Quantidade de nulos por coluna:')
print(qtd_nulos)

In [ ]:
qtd_duplicados = df_brutos['medicos'].duplicated().sum()
print('Quantidade de duplicados:', qtd_duplicados)

#### 2.3.3 Integração de Dados

Alguns países têm o nome escrito de forma diferente nas bases de saúde e no Gapminder. Corrijo os
nomes das bases de saúde para ficarem iguais aos do Gapminder (mesma ideia usada na Economia) e junto
tudo com `merge`, por país e ano.

In [ ]:
# Corrijo o nome dos países das bases de saúde para o nome usado no Gapminder
correcao_nomes_saude = {
    'Democratic Republic of Congo': 'Congo Dem. Rep.',
    'Congo': 'Congo Rep.',
    'Czechia': 'Czech Republic',
    'Hong Kong': 'Hong Kong China',
    'North Korea': 'Korea Dem. Rep.',
    'South Korea': 'Korea Rep.',
    'Slovakia': 'Slovak Republic',
    'Eswatini': 'Swaziland',
    'Yemen': 'Yemen Rep.',
    'Palestine': 'West Bank and Gaza',
}
saude['country'] = saude['country'].replace(correcao_nomes_saude)

# Junto o Gapminder (já limpo) com as bases de saúde, por país e ano
df_saude = df.merge(saude, on=['country', 'year'], how='left')
print('Linhas:', len(df_saude))
df_saude.head()

Como todas as minhas perguntas comparam os países num mesmo período, uso o ano de **2007** (o último
do Gapminder). Também crio uma coluna de **faixa de PIB** (Baixo, Médio, Alto), para conseguir
responder as perguntas que pedem para comparar países "com PIB parecido".

In [ ]:
# Filtro o ano de referência
dados_saude = df_saude[df_saude['year'] == 2007].copy()

print('Países em 2007:', len(dados_saude))
print(dados_saude[['gasto_saude', 'mortalidade', 'obesidade', 'medicos']].notna().sum())

In [ ]:
# Crio faixas de PIB (3 grupos de tamanho parecido), para as perguntas 1 e 4
corte_baixo, corte_alto = dados_saude['gdpPercap'].quantile([0.33, 0.66])

def faixa_de_pib(pib):
    if pib <= corte_baixo:
        return '1. PIB Baixo'
    elif pib <= corte_alto:
        return '2. PIB Médio'
    else:
        return '3. PIB Alto'

dados_saude['faixa_pib'] = dados_saude['gdpPercap'].apply(faixa_de_pib)
dados_saude['faixa_pib'].value_counts()

---

## 3. Funções de Visualização e Análise Gráfica

**Objetivo:** criar funções reutilizáveis para gerar gráficos e apoiar as análises, evitando repetir o mesmo código várias vezes.


| Função | Tipo de gráfico | Quando usar |
|---|---|---|
| `grafico_linha` | Linhas | Mostrar a evolução no tempo; com `hue`, compara grupos (ex.: continentes) |
| `grafico_barras` | Barras | Comparar categorias e fazer rankings (média, soma ou mediana por grupo) |
| `grafico_dispersao` | Dispersão | Ver a relação entre duas variáveis; aceita cor por grupo, tamanho por coluna, escala logarítmica e linha de tendência |
| `grafico_histograma` | Histograma | Ver a distribuição de uma variável (onde os valores se concentram) |
| `grafico_boxplot` | Boxplot | Comparar a distribuição entre grupos e identificar valores extremos |
| `grafico_correlacao` | Mapa de calor | Ver a correlação entre várias colunas de uma vez (de -1 a 1) |


In [ ]:
# Função para criar um gráfico de linhas
def grafico_linha(dados, x, y, titulo, xlabel, ylabel, figsize=(8, 5)):
    dados_grafico = dados[[x, y]].dropna().sort_values(x)

    plt.figure(figsize=figsize)
    plt.plot(dados_grafico[x], dados_grafico[y], marker='o')
    plt.title(titulo)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=45)
    plt.grid()
    plt.tight_layout()
    plt.show()

In [ ]:
# Função para criar um gráfico de barras
def grafico_barras(dados, x, y, titulo, xlabel, ylabel, horizontal=False, figsize=(8, 5)):
    dados_grafico = dados[[x, y]].dropna()

    plt.figure(figsize=figsize)

    if horizontal:
        plt.barh(dados_grafico[x], dados_grafico[y])
    else:
        plt.bar(dados_grafico[x], dados_grafico[y])
        plt.xticks(rotation=45)

    plt.title(titulo)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()

In [ ]:
# Função para criar um gráfico de dispersão
def grafico_dispersao(dados, x, y, titulo, xlabel, ylabel, figsize=(8, 5), hue=None):
    colunas = [x, y]
    if hue != None:
        colunas.append(hue)

    dados_grafico = dados[colunas].dropna()

    plt.figure(figsize=figsize)

    if hue == None:
        plt.scatter(dados_grafico[x], dados_grafico[y])
    else:
        for grupo in dados_grafico[hue].unique():
            dados_grupo = dados_grafico[dados_grafico[hue] == grupo]
            plt.scatter(
                dados_grupo[x],
                dados_grupo[y],
                label=grupo
            )

        plt.legend(title='Continente' if hue == 'continent' else hue)

    plt.title(titulo)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid()
    plt.tight_layout()
    plt.show()

In [ ]:
# Função para mostrar os 10 países que mais se destacam em um gráfico de dispersão
def tabela_destaques_dispersao(dados, x, y, coluna_destaque=None, titulo='10 países em destaque', quantidade=10):
    if coluna_destaque == None:
        coluna_destaque = y

    colunas = ['country']
    if 'continent' in dados.columns:
        colunas.append('continent')
    colunas.extend([x, y])
    colunas = list(dict.fromkeys(colunas))

    tabela = dados[colunas].dropna(subset=[x, y, coluna_destaque])
    tabela = tabela.sort_values(coluna_destaque, ascending=False).head(quantidade)

    print(titulo)
    return tabela.round(2).reset_index(drop=True)

In [ ]:
# Função para criar um histograma
def grafico_histograma(dados, coluna, titulo, xlabel, bins=15, figsize=(8, 5)):
    valores = dados[coluna].dropna()

    plt.figure(figsize=figsize)
    plt.hist(valores, bins=bins, edgecolor='black')
    plt.title(titulo)
    plt.xlabel(xlabel)
    plt.ylabel('Quantidade de registros')
    plt.tight_layout()
    plt.show()

In [ ]:
# Função para criar um boxplot
def grafico_boxplot(dados, coluna, titulo, ylabel, figsize=(8, 5)):
    valores = dados[coluna].dropna()

    plt.figure(figsize=figsize)
    plt.boxplot(valores)
    plt.title(titulo)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()

In [ ]:
def grafico_correlacao(dados, colunas, ano=None, metodo='pearson', titulo='', figsize=(6, 5)):
    d = dados if ano is None else dados[dados['year'] == ano]
    d = d[colunas].dropna()

    plt.figure(figsize=figsize)
    sns.heatmap(d.corr(method=metodo), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
    plt.title(titulo, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

---

## 4. Interpretação das Visualizações

Escolham pelo menos **15 perguntas** para investigar. Para cada análise, incluam código, tabela ou gráfico e uma interpretação escrita pelo grupo (usando, quando fizer sentido, as funções da seção 3). As perguntas abaixo são sugestões; vocês podem adaptar ou substituir por outras relacionadas ao objetivo do trabalho.

Na interpretação de cada análise, expliquem:
- O que o gráfico ou a tabela mostra e por que esse tipo de visualização foi escolhido?
- Que comparação ou tendência chamou atenção?
- A pergunta inicial foi respondida?
- Existe alguma limitação que precisa ser considerada?


#### Análise 1 — Inflação e crescimento do PIB per capita (Economia – André)
**Pergunta:** Países com maiores taxas de inflação apresentam diferenças no crescimento do PIB per capita em comparação com países com inflação mais estável?

**Visualização escolhida:** Gráficos de barras. Primeiro calculamos a média de inflação e de crescimento do PIB per capita de cada país (de 1962 a 2007, só países com pelo menos 5 anos de dados). Depois comparamos os 10 países de maior inflação com os 10 de menor inflação, usando a mediana do crescimento, porque poucos países com valores extremos distorcem a média.



In [ ]:
inf = df_economia.dropna(subset=['inflacao', 'cresc_pib'])
por_pais = inf.groupby('country')[['inflacao', 'cresc_pib']].mean()

anos = inf.groupby('country')['year'].count()
por_pais['anos'] = anos
por_pais = por_pais[por_pais['anos'] >= 5]

# Seleciona os 10 países com maior e menor inflação média
maiores = por_pais.sort_values('inflacao', ascending=False).head(10).round(2)
menores = por_pais.sort_values('inflacao').head(10).round(2)

print('países analisados:', len(por_pais), ' países')
print('10 países com maior inflação média:')
print(maiores)
print()
print('10 países com menor inflação média:')
print(menores)

In [ ]:
grafico_maiores = maiores.reset_index().sort_values('inflacao')


grafico_barras(
    dados=grafico_maiores,
    x='country',
    y='inflacao',
    titulo='10 países com maior inflação média (1962-2007)',
    xlabel='Inflação média (% ao ano)',
    ylabel='País',
    horizontal=True,
    figsize=(10, 6)
)

In [ ]:
# Identificar o grupo de cada país e os junta
maiores['grupo'] = 'Maior inflação'
menores['grupo'] = 'Menor inflação'

grupos = pd.concat([maiores, menores])

# Calcula a mediana do crescimento do PIB dos grupos
resultado = grupos.groupby('grupo')['cresc_pib'].median().reset_index()
print(resultado.round(2))

# Gráfico 2: compara o crescimento do PIB entre os grupos
grafico_barras(
    dados=resultado,
    x='grupo',
    y='cresc_pib',
    titulo='Crescimento mediano do PIB per capita: maior x menor inflação',
    xlabel='Grupo de países',
    ylabel='Crescimento do PIB per capita (% ao ano)'
)

**Interpretação do grupo:**

Entre os 10 países com maior inflação média, o crescimento mediano do PIB per capita foi de 1,80% ao ano ao longo do periodo analisado (1962 - 2007). Entre os 10 com menor inflação, foi de 2,48%. O grupo de menor inflação apresentou crescimento um pouco maior, mas valores extremos e a quantidade diferente de anos disponíveis também influenciam o resultado, porém com base nos dados disponiveis esse aumento do PIB entre paises com maior e menor inflação estão próximos da faixa visualizada.

#### Análise 2 — Comércio internacional e PIB per capita (Economia – André)
**Pergunta:** Países com maior participação do comércio internacional em suas economias apresentam maior PIB per capita?

**Visualização escolhida:** Gráfico de dispersão entre comércio (% do PIB) e PIB per capita em 2007, com cor por continente. O PIB per capita está em escala logarítmica, porque poucos países têm valores muito altos. Foi usado um único ano (2007) para comparar os países na mesma época.


In [ ]:
comercio_df = df_economia.copy()
comercio_df.loc[comercio_df['comercio'] == 0, 'comercio'] = np.nan

# Seleciona os dados de 2007
dados_comercio = comercio_df[comercio_df['year'] == 2007].copy()
dados_comercio = dados_comercio.dropna(subset=['comercio', 'gdpPercap'])

print('Países utilizados:', len(dados_comercio))

correlacao = dados_comercio['comercio'].corr(dados_comercio['gdpPercap'])
print('Correlação entre comércio e PIB per capita:', round(correlacao, 2))

In [ ]:
grafico_dispersao(
    dados=dados_comercio,
    x='comercio',
    y='gdpPercap',
    titulo='Comércio internacional x PIB per capita (2007)',
    xlabel='Comércio (% do PIB)',
    ylabel='PIB per capita',
    hue='continent'
)

In [ ]:
# Tabela: 10 países com maior PIB per capita no gráfico
tabela_destaques_dispersao(
    dados=dados_comercio,
    x='comercio',
    y='gdpPercap',
    coluna_destaque='gdpPercap',
    titulo='10 países com maior PIB per capita (2007)'
)

**Interpretação do grupo:**
Foram analisados 125 países no periodo do ano de 2007. A correlação entre comércio internacional e PIB per capita foi de 0,35, indicando uma relação positiva moderada: países com maior participação do comércio tendem a apresentar PIB per capita mais alto. Porém, há diferenças entre países e continentes, portanto o comércio, sozinho, não explica o nível de renda.

#### Análise 3 — Desemprego, crescimento e nível de renda (Economia – André)
**Pergunta:** Países com maiores taxas de desemprego apresentam menor crescimento do PIB per capita ou diferenças nos níveis de renda entre países?

**Visualização escolhida:** Dois gráficos de dispersão com os dados de 2007: desemprego x crescimento do PIB per capita, e desemprego x PIB per capita (em escala logarítmica). A base de desemprego só tem os anos de 1992, 1997, 2002 e 2007, então conferimos a correlação em cada um deles.


In [ ]:
# Selecionar os dados de 2007
dados_desemprego = df_economia[df_economia['year'] == 2007].copy()
dados_desemprego = dados_desemprego.dropna(
    subset=['desemprego', 'cresc_pib', 'gdpPercap']
)

print('Países utilizados:', len(dados_desemprego))

# Calcula as relações entre os indicadores de desemprego, crescimento do PIB e PIB per capita
correlacao_crescimento = dados_desemprego['desemprego'].corr(dados_desemprego['cresc_pib'])
correlacao_renda = dados_desemprego['desemprego'].corr(dados_desemprego['gdpPercap'])

print('Correlação entre desemprego e crescimento do PIB:', round(correlacao_crescimento, 2))
print('Correlação entre desemprego e PIB per capita:', round(correlacao_renda, 2))

In [ ]:
# desemprego e crescimento do PIB per capita
grafico_dispersao(
    dados=dados_desemprego,
    x='desemprego',
    y='cresc_pib',
    titulo='Desemprego x crescimento do PIB per capita (2007)',
    xlabel='Desemprego (% da força de trabalho)',
    ylabel='Crescimento do PIB per capita (% ao ano)',
    hue='continent'
)

In [ ]:
# 10 países com maiores taxas de desemprego
tabela_destaques_dispersao(
    dados=dados_desemprego,
    x='desemprego',
    y='cresc_pib',
    coluna_destaque='desemprego',
    titulo='10 países com maior taxa de desemprego (2007)'
)

In [ ]:
# desemprego e PIB per capita
grafico_dispersao(
    dados=dados_desemprego,
    x='desemprego',
    y='gdpPercap',
    titulo='Desemprego x PIB per capita (2007)',
    xlabel='Desemprego (% da força de trabalho)',
    ylabel='PIB per capita',
    hue='continent'
)

In [ ]:
# os mesmos países com maior desemprego e seus níveis de renda
# Assim é possível comparar desemprego e PIB per capita.
tabela_destaques_dispersao(
    dados=dados_desemprego,
    x='desemprego',
    y='gdpPercap',
    coluna_destaque='desemprego',
    titulo='PIB per capita dos 10 países com maior desemprego (2007)'
)

**Interpretação do grupo:**

Foram analisados 138 países em 2007. A correlação entre desemprego e crescimento do PIB per capita foi de 0,04, o que é muito baixo, quase nada. Já entre desemprego e PIB per capita foi de -0,15, uma relação negativa fraca. Assim, os dados não mostram uma associação linear relevante entre desemprego e crescimento, nem uma relação forte com o nível de renda.

#### Análise 4 — Diferença de expectativa de vida entre mulheres e homens e PIB per capita (Economia – André)
**Pergunta:** A diferença de expectativa de vida entre homens e mulheres aumenta ou diminui conforme o PIB per capita cresce?

**Visualização escolhida:** Um gráfico de dispersão compara o PIB per capita médio com a diferença média de expectativa de vida de cada país. Também mostramos a evolução média da expectativa de vida feminina e masculina nos anos disponíveis. Foram considerados apenas os anos em que Gapminder e os dois indicadores do Banco Mundial possuem dados, mantendo países com informações em todos esses anos para a comparação principal.

In [ ]:
dados_vida = df_economia.dropna(subset=['gdpPercap', 'vida_f', 'vida_m']).copy()
dados_vida = dados_vida[dados_vida['gdpPercap'] > 0]

# Diferença positiva significa que a expectativa de vida feminina é maior.
dados_vida['dif_vida'] = dados_vida['vida_f'] - dados_vida['vida_m']

anos_disponiveis = sorted(dados_vida['year'].unique())
print('Anos utilizados na análise:', anos_disponiveis)
print('Total de anos:', len(anos_disponiveis))
print('Quantidade de países com dados em cada ano:')
display(
    dados_vida.groupby('year')['country']
    .nunique()
    .reset_index(name='quantidade_paises')
)

# Calcula a média dos indicadores para cada país ao longo dos anos disponíveis.
resumo_vida = (
    dados_vida.groupby(['country', 'continent'])
    .agg(
        pib_medio=('gdpPercap', 'mean'),
        vida_f_media=('vida_f', 'mean'),
        vida_m_media=('vida_m', 'mean'),
        dif_vida_media=('dif_vida', 'mean'),
        anos_analisados=('year', 'nunique')
    )
    .reset_index()
)

resumo_vida = resumo_vida[
    resumo_vida['anos_analisados'] == len(anos_disponiveis)
].copy()

correlacao = resumo_vida['pib_medio'].corr(
    resumo_vida['dif_vida_media'],
    method='spearman'
)

print('Países com dados completos em todos os anos:', resumo_vida['country'].nunique())
print('Correlação de Spearman:', round(correlacao, 2))


In [ ]:
# Gráfico: PIB per capita médio x diferença média de expectativa de vida
# Cada ponto representa um país; as cores identificam os continentes.
grafico_dispersao(
    dados=resumo_vida,
    x='pib_medio',
    y='dif_vida_media',
    titulo='PIB per capita médio x diferença de expectativa de vida (1962–2007)',
    xlabel='PIB per capita médio',
    ylabel='Diferença média (mulheres - homens), em anos',
    hue='continent',
)




In [ ]:
# Compara a expectativa de vida média de mulheres e homens em cada ano disponível.
vida_por_ano = (
    dados_vida.groupby('year')[['vida_f', 'vida_m', 'dif_vida']]
    .mean()
    .reset_index()
)

plt.figure(figsize=(9, 5))
plt.plot(vida_por_ano['year'], vida_por_ano['vida_f'], marker='o', label='Mulheres')
plt.plot(vida_por_ano['year'], vida_por_ano['vida_m'], marker='o', label='Homens')
plt.title('Expectativa de vida média por sexo nos anos analisados')
plt.xlabel('Ano')
plt.ylabel('Expectativa de vida (anos)')
plt.xticks(vida_por_ano['year'], rotation=45)
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

print('Diferença média entre mulheres e homens por ano:')
display(vida_por_ano[['year', 'dif_vida']].round(2))


In [ ]:
print('Diferença média de expectativa de vida (mulheres − homens), em anos')
tabela_destaques_dispersao(
    dados=resumo_vida,
    x='pib_medio',
    y='dif_vida_media',
    coluna_destaque='dif_vida_media',
    titulo='10 países com maior diferença média de expectativa de vida (período analisado)'
)


**Interpretação do grupo:**

Foram analisados 139 países com dados completos nos 10 anos disponíveis (1962 a 2007) e comparado com a fonte do World Bank (Banco Mundial). A correlação de Spearman foi de 0,59, indicando uma associação positiva moderada: países com maior PIB per capita médio tendem a apresentar uma diferença maior de expectativa de vida a favor das mulheres. A relação é uma associação e pode ser um dos indicadores que demonstra que o aumento do PIB cause essa diferença.

### Análises — Saúde (Leandro)

Aqui respondo as quatro perguntas da minha parte, reaproveitando as funções de gráfico da seção 3.
Começo com uma visão geral da correlação entre os indicadores e, em seguida, faço uma análise para
cada pergunta.

In [ ]:
# Função auxiliar: correlação entre duas colunas DENTRO de cada faixa de PIB
def correlacao_por_faixa(dados, coluna_x, coluna_y):
    linhas = []
    for faixa in ['1. PIB Baixo', '2. PIB Médio', '3. PIB Alto']:
        grupo = dados[dados['faixa_pib'] == faixa]
        corr = grupo[coluna_x].corr(grupo[coluna_y])
        qtd = grupo[[coluna_x, coluna_y]].dropna().shape[0]
        linhas.append({'faixa_pib': faixa, 'correlacao': round(corr, 2), 'qtd_paises': qtd})
    return pd.DataFrame(linhas)

In [ ]:
# Visão geral: correlação entre todos os indicadores de saúde + vida e PIB (2007)
grafico_correlacao(
    dados_saude,
    ['lifeExp', 'gdpPercap', 'gasto_saude', 'mortalidade', 'obesidade', 'medicos'],
    titulo='Correlação entre os indicadores de saúde (2007)'
)

#### Análise 1 — Gasto em saúde e expectativa de vida (Saúde – Leandro)
**Pergunta:** Países que gastam mais em saúde por pessoa vivem mais, **mesmo quando têm PIB per capita parecido**?

**Visualização escolhida:** Gráfico de dispersão entre gasto em saúde e expectativa de vida (cor por continente), em 2007, e um gráfico de barras com a correlação dentro de cada faixa de PIB. Separei os países em 3 faixas de PIB justamente para comparar países de riqueza parecida e ver se a relação se mantém.

In [ ]:
grafico_dispersao(
    dados_saude, x='gasto_saude', y='lifeExp',
    titulo='Gasto em saúde per capita x Expectativa de vida (2007)',
    xlabel='Gasto em saúde per capita (US$ PPP)', ylabel='Expectativa de vida (anos)',
    hue='continent'
)
print('Correlação geral:', dados_saude['gasto_saude'].corr(dados_saude['lifeExp']).round(2))

In [ ]:
corr_p1 = correlacao_por_faixa(dados_saude, 'gasto_saude', 'lifeExp')
grafico_barras(
    corr_p1, x='faixa_pib', y='correlacao',
    titulo='Gasto em saúde x Expectativa de vida — por faixa de PIB',
    xlabel='Faixa de PIB', ylabel='Correlação'
)
corr_p1

**Interpretação do grupo:**

Foram analisados 133 países em 2007. A correlação geral entre gasto em saúde e expectativa de vida foi de **0,63**, uma relação positiva e forte: no geral, países que gastam mais em saúde por pessoa tendem a viver mais. Quando separo por faixa de PIB, a correlação continua positiva nas três faixas (**0,34** no PIB baixo, **0,23** no médio e **0,56** no alto), então o maior gasto em saúde está ligado a mais anos de vida **mesmo entre países de renda parecida**. A limitação é que parte desse efeito acompanha a própria riqueza do país, então não dá para dizer que é só o gasto em saúde que causa o aumento.

#### Análise 2 — Mortalidade infantil e expectativa de vida (Saúde – Leandro)
**Pergunta:** A **mortalidade infantil** explica melhor as diferenças de expectativa de vida entre os países **do que o PIB per capita**?

**Visualização escolhida:** Gráfico de dispersão entre mortalidade infantil e expectativa de vida (cor por continente) e um gráfico de barras comparando a força (valor absoluto) da correlação de cada fator com a expectativa de vida. Usei 2007 para comparar todos os países na mesma época.

In [ ]:
grafico_dispersao(
    dados_saude, x='mortalidade', y='lifeExp',
    titulo='Mortalidade infantil x Expectativa de vida (2007)',
    xlabel='Mortalidade infantil (por 1.000)', ylabel='Expectativa de vida (anos)',
    hue='continent'
)

In [ ]:
corr_mort = dados_saude['mortalidade'].corr(dados_saude['lifeExp'])
corr_pib  = dados_saude['gdpPercap'].corr(dados_saude['lifeExp'])
comparacao = pd.DataFrame({
    'fator': ['Mortalidade infantil', 'PIB per capita'],
    'forca_correlacao': [round(abs(corr_mort), 2), round(abs(corr_pib), 2)]
})
grafico_barras(
    comparacao, x='fator', y='forca_correlacao',
    titulo='Força da relação com a Expectativa de vida (2007)',
    xlabel='Fator', ylabel='Força da correlação (|r|)'
)
print('Correlação mortalidade x vida:', round(corr_mort, 2))
print('Correlação PIB x vida:', round(corr_pib, 2))

**Interpretação do grupo:**

Foram analisados 139 países em 2007. A correlação entre mortalidade infantil e expectativa de vida foi de **−0,91** (negativa e muito forte), bem mais forte em módulo que a do PIB per capita, que foi de **0,68**. Então a mortalidade infantil **explica melhor** as diferenças de expectativa de vida entre os países do que o PIB. Isso faz sentido, porque a expectativa de vida ao nascer é muito puxada para baixo quando morrem muitas crianças. Vale lembrar que os dois fatores também estão ligados entre si (países mais ricos costumam ter menos mortes infantis).

#### Análise 3 — PIB per capita e obesidade (Saúde – Leandro)
**Pergunta:** A **taxa de obesidade** cresce junto com o **PIB per capita**?

**Visualização escolhida:** Gráfico de dispersão entre PIB per capita e obesidade (cor por continente), em 2007. A dispersão ajuda a ver se os dois sobem juntos e se há países fora do padrão.

In [ ]:
grafico_dispersao(
    dados_saude, x='gdpPercap', y='obesidade',
    titulo='PIB per capita x Obesidade em adultos (2007)',
    xlabel='PIB per capita (US$)', ylabel='Obesidade em adultos (%)',
    hue='continent'
)
print('Correlação:', dados_saude['gdpPercap'].corr(dados_saude['obesidade']).round(2))

**Interpretação do grupo:**

Foram analisados 139 países em 2007. A correlação entre PIB per capita e obesidade foi de **0,53**, uma relação positiva moderada: em média, países mais ricos têm mais obesidade. Mas a relação não é tão forte, e o gráfico mostra bastante dispersão — tem país rico com obesidade baixa (como alguns da Ásia) e país de renda média com obesidade alta. Então o PIB ajuda a explicar em parte, mas hábitos alimentares e cultura também pesam bastante.

#### Análise 4 — Médicos e expectativa de vida (Saúde – Leandro)
**Pergunta:** Países com **mais médicos por mil habitantes** têm maior expectativa de vida, **mesmo descontando o efeito do PIB per capita**?

**Visualização escolhida:** Gráfico de dispersão entre médicos e expectativa de vida (cor por continente) e um gráfico de barras com a correlação dentro de cada faixa de PIB — do mesmo jeito da Análise 1, para "descontar" o efeito da riqueza.

In [ ]:
grafico_dispersao(
    dados_saude, x='medicos', y='lifeExp',
    titulo='Médicos por 1.000 habitantes x Expectativa de vida (2007)',
    xlabel='Médicos por 1.000 habitantes', ylabel='Expectativa de vida (anos)',
    hue='continent'
)
print('Correlação geral:', dados_saude['medicos'].corr(dados_saude['lifeExp']).round(2))

In [ ]:
corr_p4 = correlacao_por_faixa(dados_saude, 'medicos', 'lifeExp')
grafico_barras(
    corr_p4, x='faixa_pib', y='correlacao',
    titulo='Médicos x Expectativa de vida — por faixa de PIB',
    xlabel='Faixa de PIB', ylabel='Correlação'
)
corr_p4

**Interpretação do grupo:**

Foram analisados 83 países em 2007 (esse indicador de médicos tem menos países com dados nesse ano, então essa análise é mais limitada que as outras). A correlação geral entre médicos e expectativa de vida foi de **0,72** (forte) e, dentro de cada faixa de PIB, continua positiva (**0,54** no PIB baixo, **0,45** no médio e **0,56** no alto). Ou seja, ter mais médicos está ligado a uma expectativa de vida maior **mesmo entre países de renda parecida**.

### Síntese da parte de Saúde (Leandro)

| Pergunta | Resposta | Correlação (2007) |
|---|---|---|
| Gasto em saúde → vida (mesmo PIB) | **Sim** | 0,63 (positiva nas 3 faixas) |
| Mortalidade infantil explica melhor que o PIB? | **Sim** | −0,91 vs 0,68 |
| Obesidade cresce com o PIB? | **Sim, moderada** | 0,53 |
| Médicos → vida (descontando PIB) | **Sim** | 0,72 (positiva nas 3 faixas) |

No geral, os indicadores de saúde estão bem ligados à expectativa de vida. A **mortalidade infantil**
foi o fator mais forte, e tanto o **gasto em saúde** quanto o **número de médicos** continuam
importando **mesmo quando comparo países de PIB parecido** — ou seja, a riqueza ajuda, mas não explica
tudo sozinha. A **obesidade** foi a única que sobe junto com a riqueza (e mesmo assim de forma só
moderada).

#### Análise 5 — PIB por continente
**Pergunta:** Como o PIB per capita se distribui entre os continentes em um ano escolhido?

**Visualização possível:** Gráfico de barras ou boxplot.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [ ]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

#### Análise 6 — Expectativa de vida e PIB
**Pergunta:** Existe uma relação visual entre PIB per capita e expectativa de vida?

**Visualização possível:** Gráfico de dispersão.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [ ]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

#### Análise 7 — Países com maior expectativa de vida
**Pergunta:** Quais países apresentam os maiores valores de expectativa de vida em um ano escolhido?

**Visualização possível:** Tabela de ranking ou gráfico de barras.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [ ]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

#### Análise 8 — Países com menor expectativa de vida
**Pergunta:** Quais países apresentam os menores valores de expectativa de vida em um ano escolhido?

**Visualização possível:** Tabela de ranking ou gráfico de barras.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [ ]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

#### Análise 9 — Maiores PIBs per capita
**Pergunta:** Quais países têm os maiores PIBs per capita em um ano escolhido?

**Visualização possível:** Tabela de ranking ou gráfico de barras.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [ ]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

#### Análise 10 — Menores PIBs per capita
**Pergunta:** Quais países têm os menores PIBs per capita em um ano escolhido?

**Visualização possível:** Tabela de ranking ou gráfico de barras.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [ ]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

#### Análise 11 — Mudanças em um país
**Pergunta:** Como a expectativa de vida e o PIB per capita de um país escolhido mudaram ao longo do tempo?

**Visualização possível:** Gráfico de linhas com evolução temporal.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [ ]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

#### Análise 12 — Disparidade entre países
**Pergunta:** A diferença entre países com maior e menor expectativa de vida mudou ao longo do período?

**Visualização possível:** Comparação de valores por ano.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [ ]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

#### Análise 13 — Distribuição da expectativa de vida
**Pergunta:** Como os valores de expectativa de vida estão distribuídos em um ano escolhido?

**Visualização possível:** Histograma ou boxplot.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [ ]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

#### Análise 14 — Distribuição do PIB per capita
**Pergunta:** Como os valores de PIB per capita estão distribuídos em um ano escolhido?

**Visualização possível:** Histograma ou boxplot.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [ ]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

#### Análise 15 — População e expectativa de vida
**Pergunta:** É possível observar algum padrão entre tamanho da população e expectativa de vida?

**Visualização possível:** Gráfico de dispersão.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [ ]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

#### Análise 16 — Variação entre continentes
**Pergunta:** Qual continente apresentou maior mudança na expectativa de vida entre dois anos escolhidos?

**Visualização possível:** Tabela comparativa ou gráfico de barras.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [ ]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

#### Análise 17 — Variação do PIB por continente
**Pergunta:** Como o PIB per capita médio de cada continente variou entre dois anos?

**Visualização possível:** Gráfico de barras agrupadas ou linhas.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [ ]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

#### Análise 18 — Comparação de anos
**Pergunta:** Quais diferenças gerais aparecem ao comparar o primeiro e o último ano da base?

**Visualização possível:** Tabela comparativa e gráficos.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [ ]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

#### Análise 19 — Valores extremos
**Pergunta:** Existem países com valores muito diferentes dos demais em expectativa de vida ou PIB per capita?

**Visualização possível:** Boxplot e inspeção dos registros.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [ ]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

#### Análise 20 — Relação entre os três indicadores
**Pergunta:** Como expectativa de vida, PIB per capita e população se comportam em conjunto?

**Visualização possível:** Gráfico de dispersão; tamanho dos pontos pode representar população.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [ ]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

---

## 5. Síntese dos Principais Insights

Registrem os principais achados com base nos resultados produzidos pelo grupo.

1. **Insight 1:** [preencher com evidência da análise]
2. **Insight 2:** [preencher com evidência da análise]
3. **Insight 3:** [preencher com evidência da análise]

**Possíveis aplicações práticas:** que relação esses resultados podem ter com políticas públicas, saúde, condições econômicas ou desigualdades entre países?

---

## 6. Conclusão e Reflexão

- Quais foram os principais aprendizados?
- Quais foram as maiores dificuldades na coleta, limpeza ou análise?
- Quais limitações da base devem ser consideradas?
- O objetivo inicial foi atendido? Justifiquem com os resultados encontrados.

---

## Referências

Registrem a fonte do dataset e quaisquer outras fontes utilizadas.

- **Gapminder** — Fundação Gapminder (arquivo `gapminder_full.csv`).
- **World Development Indicators (Banco Mundial)** — https://data.worldbank.org. Indicadores usados na parte de Economia: inflação (`FP.CPI.TOTL.ZG`), comércio (`NE.TRD.GNFS.ZS`), crescimento do PIB per capita (`NY.GDP.PCAP.KD.ZG`), desemprego (`SL.UEM.TOTL.ZS`) e expectativa de vida de mulheres e homens (`SP.DYN.LE00.FE.IN` e `SP.DYN.LE00.MA.IN`). Os arquivos estão na pasta `data/Economia`, entregue junto com o trabalho.

## Checklist antes da entrega
- [ ] As seis partes recomendadas no enunciado foram contempladas.
- [ ] As bases externas foram indicadas e entregues junto com o trabalho.
- [ ] Os valores ausentes foram removidos.
- [ ] Os registros duplicados foram eliminados.
- [ ] Foram realizadas pelo menos 15 análises.
- [ ] Cada visualização tem título, identificação dos eixos e interpretação.
- [ ] As decisões e observações estão documentadas em Markdown.
- [ ] O notebook foi executado do início ao fim e revisado.
- [ ] As análises e interpretações refletem o trabalho autoral do grupo.

**Atenção:** o enunciado proíbe o uso de IA generativa para geração automática de código ou análise. Utilizem este arquivo apenas como estrutura de organização e desenvolvam as soluções e interpretações por conta própria.